> **제10회 기출변형 문제** — 교재 506~532쪽
> 문제 506~508쪽 / 정답·해설 509~532쪽
>
> 노트북과 같은 위치에 데이터 파일(csv)을 두고 위에서부터 차례대로 실행하세요.

## 문제

제공된 학습용 데이터(train.csv)는 공공기관 건물 및 시설 현황에 따른 수도 사용량 데이터이다. 학습용 데이터를 활용하여 시설별 수도 사용량(water_usage)을 예측하는 모형을 개발하고, 모형을 평가용 데이터(test.csv)에 적용하여 수도 사용량을 예측하시오. 예측 결과는 아래의 [유의사항]과 [제출 형식]을 준수하여, CSV 파일(result.csv)을 생성하는 코드를 제출하시오.

**[유의사항]**
① 예측 결과는 RMSE(Root Mean Squared Error) 평가지표에 따라 평가함
② 타겟 변수(water_usage)는 일부 값이 0으로 기재되어 있으며, 이는 결측치를 대체한 값이다

**[제출 형식]**
① CSV 파일명: result.csv
② 예측 수도사용량 컬럼명: pred
③ 제출 컬럼 개수: pred 컬럼 1개
④ 평가용 데이터 개수와 예측 결과 데이터 개수 일치: 1,450개

**[제공 데이터]**
- train.csv: 학습용 데이터, 약 3,200개
- test.csv: 평가용 데이터, 약 1,450개 (water_usage 컬럼 미제공)

| 컬럼명 | 설명 |
|---|---|
| id | 시설 ID |
| building_area | 건물 연면적(㎡) |
| resident_count | 상주 인원 수 |
| business_count | 입주 사업체 수 |
| park_area_ratio | 인근 공원 면적 비율 |
| avg_building_age | 평균 건축 연수(년) |
| season | 측정 계절 (봄/여름/가을/겨울) |
| water_usage | (train에만 존재) 수도 사용량(톤) |

### 문제 풀이 방향

이 문제에서 가장 먼저 신경 써야 할 것은 유의사항 ②번입니다. water_usage 중 일부가 0으로 기재되어 있는데, 이는 실제로 수도 사용량이 0이라는 뜻이 아니라 결측치를 0으로 표시해둔 것이라고 명시되어 있습니다. 이걸 그냥 지나치고 0을 진짜 사용량 값으로 취급해서 학습에 사용하면, 모델은 "이런 조건이면 수도 사용량이 0톤"이라는 잘못된 패턴을 배우게 됩니다. 그래서 본격적인 모델링에 앞서 이 0값들을 어떻게 다룰지 먼저 판단해야 합니다. 또한 season처럼 순서나 크기 관계가 없는 범주형 변수를 다루는 방법으로 get_dummies(원-핫 인코딩)를 사용하겠습니다.

### 정답 코드 및 해설

**셀 1) 데이터 불러오기 + 구조 확인**

In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(train.shape, test.shape)
print(train.head())

(3200, 8) (1450, 7)
   id  building_area  resident_count  business_count  park_area_ratio  avg_building_age season  water_usage
0   1        11504.8            1440             151            0.222                13      봄       1663.3
1   2         2917.6            2426              15            0.053                12     여름       2460.6
2   3        10495.3            2360             245            0.231                18     겨울       2733.8
3   4         4994.5             354              56            0.085                 4      봄        552.2
4   5          858.8            1792              19            0.037                38     가을       1723.0


train은 8개 컬럼(water_usage 포함), test는 7개 컬럼(water_usage 없음)입니다. season을 제외한 모든 독립변수가 이미 숫자형입니다.

**셀 2) 타겟 변수 확인 (0값 존재 여부 포함)**

In [2]:
print(train.isna().sum().sum(), test.isna().sum().sum())
print(train['water_usage'].describe())
print("0으로 기재된 행 수:", (train['water_usage'] == 0).sum())

0 0
count    3200.000000
mean     1512.855875
std       738.705233
min         0.000000
25%       944.575000
50%      1529.750000
75%      2115.050000
max      3211.100000
Name: water_usage, dtype: float64
0으로 기재된 행 수: 144


일반적인 결측치(NaN)는 없지만, `describe()`의 최솟값이 0이고 실제로 144개 행이 정확히 0이라는 것을 확인할 수 있습니다. 수도 사용량 평균이 1,500톤대인 것을 생각하면, 0톤짜리 시설이 144건이나 있다는 것은 상식적으로 이상합니다. 유의사항에서 이미 이 값들이 결측치 대체값이라고 알려주었으므로, 이 144건은 진짜 데이터가 아니라 사실상 결측치라고 판단하는 것이 타당합니다.

**셀 3) 결측치로 판단되는 0값 행을 학습에서 제외**

In [3]:
train_clean = train[train['water_usage'] > 0].reset_index(drop=True)
print("제외 전:", train.shape, "제외 후:", train_clean.shape)

제외 전: (3200, 8) 제외 후: (3056, 8)


`train['water_usage'] > 0`으로 수도 사용량이 0보다 큰 행만 남기고, 결측치가 대체된 것으로 보이는 144개 행은 학습 데이터에서 제외했습니다. 이 처리는 학습(train) 데이터에만 적용합니다. test 데이터는 애초에 water_usage 컬럼 자체가 제공되지 않으므로 이런 판단이 필요하지 않습니다.

> 💡 문제의 유의사항에 타겟 변수나 특정 컬럼에 대한 데이터 품질 관련 안내(0으로 대체됨, 특정 값은 오류임 등)가 있다면, 그 값을 무비판적으로 그대로 사용하지 말고 어떻게 처리할지 먼저 판단하는 것이 중요합니다. 이런 안내는 대체로 문제를 푸는 사람이 반드시 짚고 넘어가야 할 지점을 알려주는 힌트입니다.

**셀 4) 범주형 변수 원-핫 인코딩 (get_dummies)**

In [4]:
cat_cols = ['season']
train_dum = pd.get_dummies(train_clean, columns=cat_cols)
test_dum = pd.get_dummies(test, columns=cat_cols)

print(train_dum.columns.tolist())

['id', 'building_area', 'resident_count', 'business_count', 'park_area_ratio', 'avg_building_age', 'water_usage', 'season_가을', 'season_겨울', 'season_봄', 'season_여름']


`get_dummies()`는 범주형 컬럼 하나를, 그 컬럼에 등장하는 범주 개수만큼의 새로운 컬럼으로 펼쳐줍니다. season(봄/여름/가을/겨울)은 `season_봄`, `season_여름`, `season_가을`, `season_겨울`이라는 4개의 True/False 컬럼으로 바뀝니다. 계절처럼 봄과 여름 사이에 크고 작음이라는 개념이 없는 순수한 명목형 변수는, 순서가 있는 정수로 바꾸는 LabelEncoder보다 각 범주를 독립적인 컬럼으로 분리하는 get_dummies가 더 적합합니다.

**셀 5) train과 test의 컬럼 구성 맞추기**

In [5]:
test_dum = test_dum.reindex(columns=train_dum.drop(columns=['water_usage']).columns, fill_value=0)

print(test_dum.shape)
print(test_dum.head())

(1450, 10)
     id  building_area  resident_count  business_count  park_area_ratio  avg_building_age  season_가을  season_겨울  season_봄  season_여름
0  3201         9281.0             992             131            0.185                 1      False      False      True      False
1  3202        13862.8            1293             271            0.222                 6       True      False     False      False
2  3203         1650.7            2129             140            0.385                23      False      False      True      False
3  3204        12721.1            1805             271            0.189                 1      False      False     False       True
4  3205         5073.9            1996             218            0.284                29      False      False      True      False


get_dummies를 train과 test에 각각 따로 적용하면, 혹시라도 한쪽에는 등장하지만 다른 쪽에는 등장하지 않는 범주가 있을 경우 두 데이터의 컬럼 구성이 서로 달라질 수 있습니다. `reindex(columns=..., fill_value=0)`는 test_dum의 컬럼을 train_dum과 동일한 목록, 동일한 순서로 강제로 맞춰줍니다. train에는 있지만 test에는 없던 컬럼은 값이 0(해당 범주가 아님)으로 채워져 새로 생기고, 순서도 train과 정확히 같아집니다.

**셀 6) 학습용/검증용 데이터 분리**

In [6]:
from sklearn.model_selection import train_test_split

X = train_dum.drop(columns=['id', 'water_usage'])
y = train_dum['water_usage']
X_test = test_dum.drop(columns=['id'])

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_tr.shape, X_val.shape)

(2444, 9) (612, 9)


`id`는 시설을 구분하기 위한 값일 뿐 수도 사용량과 직접적인 관련이 없으므로 학습에 사용할 컬럼(X)에서 제외했습니다. `train_test_split`으로 train_dum의 일부(20%)를 검증용으로 떼어둔 이유는, 진짜 test.csv에는 정답(water_usage)이 없어서 우리가 만든 모델이 얼마나 정확한지 미리 확인할 방법이 없기 때문입니다. 학습 데이터 중 일부를 검증용으로 남겨두면, 그 부분에 대해서는 정답을 알고 있으므로 모델의 예측과 실제값을 비교해서 점수를 미리 가늠해볼 수 있습니다.

**셀 7) 모델 학습 및 검증 성능 확인**

In [7]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

model = RandomForestRegressor(n_estimators=300, random_state=42)
model.fit(X_tr, y_tr)

val_pred = model.predict(X_val)
print("검증 RMSE:", mean_squared_error(y_val, val_pred) ** 0.5)

검증 RMSE: 111.59035203138


`mean_squared_error`로 오차의 제곱평균(MSE)을 구한 뒤 `** 0.5`로 제곱근을 씌워 RMSE를 계산했습니다. 검증 RMSE가 약 111.6이라는 것은, 평균적으로 예측이 실제 수도 사용량과 약 112톤 정도 차이 난다는 뜻이며, 수도 사용량 평균이 1,500톤대인 것을 고려하면 준수한 수준입니다.

**셀 8) 전체 학습 데이터로 재학습 후 test 예측**

In [8]:
model.fit(X, y)
pred = model.predict(X_test)

print(pred[:10])

[1394.75166667 1888.52       2157.438      2421.70966667 2266.34
 1148.87366667 2122.73833333  975.602      1843.80833333 1686.49166667]


검증까지 마친 뒤에는 train_clean 전체(0값을 제외한 3,056건)를 사용해 다시 학습시켰습니다. RandomForest 계열 모델은 훈련 데이터에서 관찰한 값의 범위 안에서 예측하는 특성이 있어, 별도의 음수 보정 없이도 예측값이 상식적인 범위(양수) 안에 있는 것을 확인할 수 있습니다.

**셀 9) 제출 파일 생성**

In [9]:
result = pd.DataFrame({'pred': pred})
result.to_csv('result.csv', index=False)

print(result.shape)
print(result.head())

(1450, 1)
          pred
0  1394.751667
1  1888.520000
2  2157.438000
3  2421.709667
4  2266.340000


행 개수가 요구된 1,450개와 정확히 일치하는지 확인했습니다.

### 전체 코드

In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# 결측치가 0으로 대체된 것으로 판단되는 행을 학습에서 제외
train_clean = train[train['water_usage'] > 0].reset_index(drop=True)

# 범주형 변수 원-핫 인코딩
cat_cols = ['season']
train_dum = pd.get_dummies(train_clean, columns=cat_cols)
test_dum = pd.get_dummies(test, columns=cat_cols)

# test의 컬럼을 train 기준으로 맞추기
test_dum = test_dum.reindex(columns=train_dum.drop(columns=['water_usage']).columns, fill_value=0)

X = train_dum.drop(columns=['id', 'water_usage'])
y = train_dum['water_usage']
X_test = test_dum.drop(columns=['id'])

# 검증용 분리 후 성능 확인
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=300, random_state=42)
model.fit(X_tr, y_tr)
val_pred = model.predict(X_val)
print("검증 RMSE:", mean_squared_error(y_val, val_pred) ** 0.5)

# 전체 데이터로 재학습 후 test 예측
model.fit(X, y)
pred = model.predict(X_test)

# 제출 파일 생성
result = pd.DataFrame({'pred': pred})
result.to_csv('result.csv', index=False)
print(result.shape)

검증 RMSE: 111.59035203138
(1450, 1)


### 정리

이 문제의 핵심은 두 가지였습니다. 첫째는 유의사항에 적힌 "0은 결측치를 대체한 값"이라는 문장을 무시하지 않고, 그 값을 학습에서 제외하는 판단을 내린 것입니다. 둘째는 get_dummies로 순서 없는 범주형 변수를 인코딩하면서, train과 test의 컬럼 구성을 reindex로 맞춰야 한다는 것이었습니다.

이번 모델의 검증 RMSE는 약 111.6 수준이었습니다.

### 🔍 실전 대비: 예상되는 변형과 접근 전략

기출 복원 자료에는 "타겟 변수 일부가 0으로 결측 대체됨"이라는 힌트가 여러 회차에서 반복적으로 등장했습니다. 이는 우연이 아니라, 실제 공공데이터에서 결측치를 0으로 채워 배포하는 경우가 흔하기 때문입니다. 평가지표 자체는 RMSE가 아니라 MAE로 바뀌어 나올 수 있으므로, 두 지표를 모두 계산할 수 있어야 합니다.

**변형 예시: 평가지표가 RMSE가 아니라 MAE라면**

In [11]:
from sklearn.metrics import mean_absolute_error

# 검증 전용 모델(X_tr로만 학습)을 다시 만들어 비교합니다.
# 셀 8에서 model을 전체 데이터(X, y)로 재학습시켰기 때문에,
# 그 상태 그대로 X_val을 예측하면 모델이 이미 검증 데이터를 학습에 사용한 뒤라
# 점수가 실제보다 낮게(좋게) 나오는 문제가 생깁니다.
model_for_val = RandomForestRegressor(n_estimators=300, random_state=42)
model_for_val.fit(X_tr, y_tr)

val_pred_mae = model_for_val.predict(X_val)
print("검증 MAE:", mean_absolute_error(y_val, val_pred_mae))

검증 MAE: 87.62799019607853


`mean_squared_error` 대신 `mean_absolute_error`를 불러와서 그대로 대입하면 되지만, 여기서 한 가지 반드시 주의해야 할 점이 있습니다. 셀 8에서 이미 `model`을 train 전체(X, y)로 재학습시켜뒀기 때문에, 그 시점 이후에 `model.predict(X_val)`을 하면 모델이 이미 X_val을 학습 과정에서 본 뒤이므로 검증 점수가 실제보다 좋게(낮게) 나오는 데이터 누수(data leakage)가 발생합니다. 그래서 `model_for_val`이라는 이름으로 X_tr, y_tr만으로 다시 학습시킨 새 모델을 만들어 비교했습니다. RMSE(111.6)와 MAE(87.6)가 서로 다른 값으로 나오는 이유는, RMSE는 오차를 제곱하기 때문에 몇 개의 큰 오차(이상치)에 더 민감하게 반응하는 반면, MAE는 모든 오차를 절댓값으로 동일한 비중으로 반영하기 때문입니다.

> 💡 검증(validation) 목적으로 어떤 지표를 다시 계산하고 싶을 때는, 그 시점에 사용하는 모델이 검증 데이터를 이미 학습에 포함시킨 상태는 아닌지 항상 확인하는 습관이 필요합니다. 재학습 이후의 모델 변수를 재사용하면 이런 데이터 누수가 소리 없이 발생할 수 있습니다.

---

## 이번 회차에서 사용한 핵심 개념 정리
| 상황 | 함수/개념 | 핵심 이유 |
|---|---|---|
| 타겟 변수에 결측치가 특정 값(0 등)으로 대체되어 있을 때 | 해당 행을 학습에서 제외 (`df[df[col] > 0]`) | 잘못된 값을 그대로 학습하면 모델이 왜곡됨 |
| 순서 없는(명목형) 범주형 변수 인코딩 | `pd.get_dummies(df, columns=[...])` | 범주 간 크기 관계가 없다고 모델에 알려주는 인코딩 방식 |
| get_dummies 사용 후 train/test 컬럼 정렬 | `test.reindex(columns=train.columns, fill_value=0)` | 양쪽에 등장하는 범주가 다르면 컬럼 개수가 어긋날 수 있음 |
| 회귀 평가지표 계산(RMSE) | `mean_squared_error(y, pred) ** 0.5` | 오차 제곱 평균에 제곱근을 씌워 원래 단위로 해석 |
| 회귀 평가지표 계산(MAE) | `mean_absolute_error(y, pred)` | 오차를 절댓값으로만 반영, 이상치에 상대적으로 덜 민감 |